<a href="https://colab.research.google.com/github/U-mar-404/AIPDD-Lab05/blob/main/Week05_StudentID_231243.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 05 Lab: Text Embeddings, Semantic Retrieval and CLIP
**Name: Muhammad Umar**  
**Student ID: 231243**

## Setup

In [1]:
!pip -q install sentence-transformers transformers torch pillow pandas

In [2]:
import numpy as np
import pandas as pd
import torch
from PIL import Image
from sentence_transformers import SentenceTransformer, util

text_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Task 1: Semantic similarity

In [3]:
utterances = [
    "Turn on the classroom fan",
    "Please switch the fan on",
    "Start the ventilation fan",
    "Turn off the classroom fan",
    "Switch on the projector",
    "Submit my final year project report",
    "Upload the FYP document"
]

emb = text_model.encode(utterances, convert_to_tensor=True)
sim = util.cos_sim(emb, emb)

print("Embedding shape:", emb.shape)
print("Similarity shape:", sim.shape)
print(sim.cpu().numpy().round(3))

Embedding shape: torch.Size([7, 384])
Similarity shape: torch.Size([7, 7])
[[ 1.     0.563  0.527  0.928  0.281  0.076  0.02 ]
 [ 0.563  1.     0.596  0.518  0.408 -0.012  0.016]
 [ 0.527  0.596  1.     0.49   0.168  0.056 -0.028]
 [ 0.928  0.518  0.49   1.     0.254  0.044  0.   ]
 [ 0.281  0.408  0.168  0.254  1.     0.171 -0.032]
 [ 0.076 -0.012  0.056  0.044  0.171  1.     0.147]
 [ 0.02   0.016 -0.028  0.    -0.032  0.147  1.   ]]


**T1.1** The embedding matrix has shape (7, 384). The 7 rows are the 7 sentences. The 384 columns are the dimensions of the embedding space. Each sentence is one point (vector) in that space. A single coordinate has no readable meaning, and the meaning comes from the whole vector.

In [4]:
def top_matches(query, texts, k=3):
    q = text_model.encode(query, convert_to_tensor=True)
    t = text_model.encode(texts, convert_to_tensor=True)
    scores = util.cos_sim(q, t)[0]
    top = torch.topk(scores, k=k)
    return [(texts[i], round(float(s), 3)) for s, i in zip(top.values, top.indices)]

print("T1.2 Top 3 for:", utterances[0])
for text, score in top_matches(utterances[0], utterances[1:], k=3):
    print(score, "-", text)

print()
print("T1.3")
print("Turn on vs Turn off:", round(float(sim[0][3]), 3))
print("Turn on vs Please switch the fan on:", round(float(sim[0][1]), 3))

T1.2 Top 3 for: Turn on the classroom fan
0.928 - Turn off the classroom fan
0.563 - Please switch the fan on
0.527 - Start the ventilation fan

T1.3
Turn on vs Turn off: 0.928
Turn on vs Please switch the fan on: 0.563


**T1.3** "Turn on" and "Turn off" get a high score because they share the same topic (the classroom fan) and almost the same words. The model measures how related the meaning is, not whether the action is the same. A system that turns machines on or off needs extra logic to separate opposite intents.

**T1.5** Own sentences. Prediction first, then the result.

In [5]:
new_utterances = [
    "Kindly power up the fan in the room",
    "Begin the ventilation system",
    "Do not turn on the classroom fan"
]

print("Prediction: the first two should be closest to the fan-on commands.")
print("The negative command will probably also be very close to 'Turn on the classroom fan'.")
print()

for s in new_utterances:
    print("Query:", s)
    for text, score in top_matches(s, utterances, k=3):
        print("  ", score, "-", text)

Prediction: the first two should be closest to the fan-on commands.
The negative command will probably also be very close to 'Turn on the classroom fan'.

Query: Kindly power up the fan in the room
   0.613 - Please switch the fan on
   0.601 - Start the ventilation fan
   0.562 - Turn on the classroom fan
Query: Begin the ventilation system
   0.786 - Start the ventilation fan
   0.25 - Please switch the fan on
   0.231 - Turn on the classroom fan
Query: Do not turn on the classroom fan
   0.939 - Turn on the classroom fan
   0.926 - Turn off the classroom fan
   0.596 - Please switch the fan on


**T1.5 Observation:** (Write here whether your prediction matched the output. The negative command usually still ranks close to the "Turn on" sentence.)

## Task 2: Semantic FAQ retriever

In [6]:
faq = [
    ("FYP registration procedure", "Submit the approved FYP registration form to the coordinator."),
    ("Course withdrawal policy", "Contact the academic office and follow the withdrawal procedure."),
    ("Lab equipment booking", "Request a laboratory equipment booking through the lab office."),
    ("Internship application process", "Follow the department internship application procedure."),
    ("Project report submission", "Submit your project report according to the announced deadline.")
]

titles = [item[0] for item in faq]
faq_embeddings = text_model.encode(titles, convert_to_tensor=True)

def retrieve_faq(question, threshold=0.45):
    if not question or not question.strip():
        return {"status": "empty", "answer": None, "score": None}

    query_vec = text_model.encode(question, convert_to_tensor=True)
    scores = util.cos_sim(query_vec, faq_embeddings)[0]

    best = int(scores.argmax())
    score = float(scores[best])

    if score < threshold:
        return {"status": "no_match", "answer": None, "score": round(score, 3)}

    return {
        "status": "ok",
        "title": titles[best],
        "answer": faq[best][1],
        "score": round(score, 3)
    }

**T2.1 and T2.2** Test questions, including paraphrases and an unrelated query.

In [7]:
questions = [
    "How do I register for my FYP?",
    "I want to drop a course",
    "How can I reserve equipment in the lab?",
    "Where do I apply for an internship?",
    "When is the deadline to hand in the project report?",
    "What is the weather in Lahore?",
    ""
]

for q in questions:
    print("Q:", repr(q))
    print("  ", retrieve_faq(q))

Q: 'How do I register for my FYP?'
   {'status': 'ok', 'title': 'FYP registration procedure', 'answer': 'Submit the approved FYP registration form to the coordinator.', 'score': 0.789}
Q: 'I want to drop a course'
   {'status': 'ok', 'title': 'Course withdrawal policy', 'answer': 'Contact the academic office and follow the withdrawal procedure.', 'score': 0.534}
Q: 'How can I reserve equipment in the lab?'
   {'status': 'ok', 'title': 'Lab equipment booking', 'answer': 'Request a laboratory equipment booking through the lab office.', 'score': 0.698}
Q: 'Where do I apply for an internship?'
   {'status': 'ok', 'title': 'Internship application process', 'answer': 'Follow the department internship application procedure.', 'score': 0.759}
Q: 'When is the deadline to hand in the project report?'
   {'status': 'ok', 'title': 'Project report submission', 'answer': 'Submit your project report according to the announced deadline.', 'score': 0.664}
Q: 'What is the weather in Lahore?'
   {'status

**T2.3 and T2.4** Test set, threshold comparison and accuracy.

In [8]:
test_set = [
    ("How do I register for my FYP?", "FYP registration procedure"),
    ("What is the process to start my final year project?", "FYP registration procedure"),
    ("I want to drop a course", "Course withdrawal policy"),
    ("How can I leave a subject after the semester starts?", "Course withdrawal policy"),
    ("How can I reserve equipment in the lab?", "Lab equipment booking"),
    ("Where do I apply for an internship?", "Internship application process"),
    ("When is the deadline to hand in the project report?", "Project report submission"),
    ("What is the weather in Lahore?", None),
    ("Who won the cricket match yesterday?", None),
    ("Give me a recipe for biryani", None)
]

in_scope = [t for t in test_set if t[1] is not None]
out_scope = [t for t in test_set if t[1] is None]

correct = 0
for q, expected in in_scope:
    result = retrieve_faq(q, threshold=0.0)
    if result["title"] == expected:
        correct += 1
print("Top-1 accuracy on in-scope questions:", correct, "/", len(in_scope))

print()
rows = []
for th in [0.30, 0.45, 0.60]:
    accepted_in = 0
    rejected_out = 0
    for q, expected in in_scope:
        if retrieve_faq(q, threshold=th)["status"] == "ok":
            accepted_in += 1
    for q, expected in out_scope:
        if retrieve_faq(q, threshold=th)["status"] == "no_match":
            rejected_out += 1
    rows.append({
        "threshold": th,
        "in-scope accepted": str(accepted_in) + "/" + str(len(in_scope)),
        "out-of-scope rejected": str(rejected_out) + "/" + str(len(out_scope))
    })
print(pd.DataFrame(rows))

print()
for q, expected in test_set:
    r = retrieve_faq(q, threshold=0.0)
    print(r["score"], "|", q, "->", r["title"])

Top-1 accuracy on in-scope questions: 6 / 7

   threshold in-scope accepted out-of-scope rejected
0       0.30               7/7                   3/3
1       0.45               5/7                   3/3
2       0.60               4/7                   3/3

0.789 | How do I register for my FYP? -> FYP registration procedure
0.438 | What is the process to start my final year project? -> Project report submission
0.534 | I want to drop a course -> Course withdrawal policy
0.421 | How can I leave a subject after the semester starts? -> Course withdrawal policy
0.698 | How can I reserve equipment in the lab? -> Lab equipment booking
0.759 | Where do I apply for an internship? -> Internship application process
0.664 | When is the deadline to hand in the project report? -> Project report submission
0.086 | What is the weather in Lahore? -> Project report submission
0.055 | Who won the cricket match yesterday? -> Lab equipment booking
0.081 | Give me a recipe for biryani -> Internship applica

**T2.5 Improvement:** Add richer FAQ text. For example, store several example questions for each FAQ entry (titles, synonyms and short descriptions) and compare the query with all of them. This gives better matches for paraphrases than a short title alone. Asking the student to clarify when the score is borderline is another option.

**Note:** Cosine similarity is not a probability. The best match can be wrong, so thresholds must be checked on real test questions.

## Task 3: CLIP zero-shot image-text matching

Upload at least four non-sensitive images (different scenes or objects, one ambiguous). In Colab, run the next cell and choose the files. If you run locally, put the images in the notebook folder.

In [9]:
from google.colab import files
uploaded = files.upload()
image_files = list(uploaded.keys())
print(image_files)

Saving images.jpg to images.jpg
['images.jpg']


In [10]:
from transformers import pipeline

clip_classifier = pipeline(
    task="zero-shot-image-classification",
    model="openai/clip-vit-base-patch32",
    device=-1
)

labels = [
    "a photograph of laboratory equipment",
    "a photograph of a classroom",
    "a photograph of food waste",
    "a photograph of a plastic bottle"
]

def classify(path, candidate_labels):
    image = Image.open(path).convert("RGB")
    return clip_classifier(image, candidate_labels=candidate_labels)

print("T3.1")
for path in image_files:
    results = classify(path, labels)
    print(path, "->", results[0]["label"], round(results[0]["score"], 3))

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

T3.1
images.jpg -> a photograph of laboratory equipment 0.995


**T3.2** Reword two prompts and compare the same images.

In [11]:
labels_reworded = [
    "a picture of science lab tools",
    "a photo of a lecture room with desks",
    "a photograph of food waste",
    "a photograph of a plastic bottle"
]

for path in image_files[:2]:
    print(path)
    print(" original:")
    for r in classify(path, labels):
        print("   ", round(r["score"], 3), r["label"])
    print(" reworded:")
    for r in classify(path, labels_reworded):
        print("   ", round(r["score"], 3), r["label"])

images.jpg
 original:
    0.995 a photograph of laboratory equipment
    0.004 a photograph of a classroom
    0.0 a photograph of a plastic bottle
    0.0 a photograph of food waste
 reworded:
    0.993 a picture of science lab tools
    0.005 a photograph of a plastic bottle
    0.001 a photo of a lecture room with desks
    0.001 a photograph of food waste


**T3.3** Add a fifth candidate description.

In [12]:
labels_five = labels + ["a photograph of a laptop computer"]

for path in image_files:
    print(path)
    for r in classify(path, labels_five):
        print("   ", round(r["score"], 3), r["label"])

images.jpg
    0.955 a photograph of laboratory equipment
    0.04 a photograph of a laptop computer
    0.004 a photograph of a classroom
    0.0 a photograph of a plastic bottle
    0.0 a photograph of food waste


**T3.3 Observation:** CLIP scores add up to 1 across the labels it is given. A new label takes some score from the others, so the old scores change. Scores are relative to the candidate set.

**T3.4** Folder decision rule.

In [13]:
folder_map = {
    "a photograph of laboratory equipment": "Lab",
    "a photograph of a classroom": "Classroom",
    "a photograph of food waste": "Waste",
    "a photograph of a plastic bottle": "Recycling"
}

def decide_folder(results, min_score=0.50):
    top = results[0]
    if top["score"] < min_score:
        return "Review"
    return folder_map.get(top["label"], "Review")

for path in image_files:
    results = classify(path, labels)
    print(path, "->", decide_folder(results, min_score=0.50))

images.jpg -> Lab


**T3.4** CLIP only gives scores. A separate program has to read the result and move the file, and the 0.50 threshold is an experimental value that needs testing. An ambiguous image can still get a confident wrong score.

**T3.5 CLIP vs YOLO:** CLIP is not the right tool for counting every object or drawing bounding boxes. It compares a whole image with text and gives one score per label, with no object locations. YOLO is an object detector. It returns a box and a class for each object, so it can count and localise.

## Task 4: Engineering test and reflection

In [14]:
print("Empty query:", retrieve_faq(""))
print("Spaces only:", retrieve_faq("   "))
print("Paraphrase:", retrieve_faq("How can I reserve equipment in the lab?"))
print("Out of domain:", retrieve_faq("What is the weather in Lahore?"))

Empty query: {'status': 'empty', 'answer': None, 'score': None}
Spaces only: {'status': 'empty', 'answer': None, 'score': None}
Paraphrase: {'status': 'ok', 'title': 'Lab equipment booking', 'answer': 'Request a laboratory equipment booking through the lab office.', 'score': 0.698}
Out of domain: {'status': 'no_match', 'answer': None, 'score': 0.086}


| Test case | Expected behavior | Observed behavior |
| --- | --- | --- |
| Empty FAQ query | Handled without exception | Returned status "empty" with no error |
| Relevant FAQ paraphrase | Returns intended topic | (write result and score) |
| Out-of-domain question | Rejected or flagged when appropriate | (write result and score) |
| Ambiguous CLIP image | Prediction and uncertainty discussed | (write top label, score and comments) |
| Changed CLIP prompt set | Ranking/score comparison recorded | (write what changed) |

**Q1. Why is semantic similarity not enough for a safety-critical action?**
Similarity only shows that two sentences are about the same topic. "Turn on" and "Turn off" score as very similar although they mean opposite actions. For machinery, the system also needs exact rules, confirmation and safety checks.

**Q2. Embedding model vs application logic?**
The embedding model only turns text into a vector. The application logic uses these vectors to take decisions, such as picking the best match, applying a threshold, rejecting a query or asking for clarification.

**Q3. In what sense is CLIP zero-shot?**
It can classify with new labels, given as text, without training a new classifier for them. It does not mean CLIP had no training. It was pretrained on a very large set of image-text pairs.

**Q4. One failure case and a mitigation.**
(Write your own observed failure, for example an out-of-domain question matched an FAQ with a score above the threshold, or an ambiguous image got a confident label.) Mitigation: tune the threshold on more test queries, add richer FAQ text, or send low-confidence cases to a human "Review" step.

## Conclusion

In this lab, sentence embeddings from all-MiniLM-L6-v2 gave sensible similarity scores for paraphrased commands and FAQ questions. A semantic FAQ retriever worked better than keyword matching, since paraphrases were matched to the right topic. However, similar scores were also given to commands with opposite meaning ("turn on" and "turn off"), so similarity alone is unsafe for control actions. The threshold experiments showed a trade-off: a low threshold accepted unrelated questions, and a high threshold rejected valid paraphrases. Cosine similarity is not a probability, so thresholds must be tested on representative queries.

CLIP classified images using text descriptions without any extra training. Its results changed when the prompt wording changed or when a new label was added, because its scores are relative to the given label set. Ambiguous images were a weakness, and CLIP cannot count objects or give bounding boxes, so a detector like YOLO is needed for that.

The main limits were the small test sets and the few images used. Possible improvements are richer FAQ text, more test queries for threshold tuning, a "Review" step for low-confidence cases, and human confirmation before any real action.